In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

torch_device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
pipeline_device = torch_device if torch_device.type == "mps" else -1
print("device:", torch_device)


device: mps


In [2]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="zero_shot_nli_pipeline_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [3]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
model_name = "typeform/distilbert-base-uncased-mnli"

classifier = pipeline(
    "zero-shot-classification",
    model=model_name,
    device=pipeline_device,
)

print(model_name)
print(classifier.model.config.id2label)



---[ TableVault Record ]---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli
{0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}
---[ TableVault Record ]---



In [5]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])



---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
---[ TableVault Record ]---



In [6]:
sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
sequences = [f"Sentence 1: {s1}\nSentence 2: {s2}" for s1, s2 in zip(sent1, sent2)]

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print(sequences[0])



---[ TableVault Record ]---
num_examples: 408
positive_rate: 0.6838235294117647
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .
---[ TableVault Record ]---



In [7]:
candidate_labels = ["paraphrase", "not paraphrase"]
hypothesis_template = "These two sentences are {}."
label_to_id = {"not paraphrase": 0, "paraphrase": 1}

batch_size = 16
all_outputs = []
vault.create_record_list("mrpc_zero_shot_prediction_value", column_names=["prediction"])

for i in tqdm(range(0, len(sequences), batch_size)):
    batch_sequences = sequences[i:i + batch_size]
    batch_outputs = classifier(
        batch_sequences,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False,
        batch_size=batch_size,
        truncation=True,
    )
    all_outputs.extend(batch_outputs)

y_pred = np.array([label_to_id[out["labels"][0]] for out in all_outputs])
print("done")



---[ TableVault Record ]---


  0%|          | 0/26 [00:00<?, ?it/s]

done
---[ TableVault Record ]---



In [8]:
for i in range(len(y_pred)):
    vault.append_record("mrpc_zero_shot_prediction_value", {"prediction": int(y_pred[i])}, 
                       input_items = {"glue_mrpc_validation": [i, i + 1]}
                       )

description = "Per-example zero-shot classification outputs for the GLUE MRPC validation set, generated by the Hugging Face zero-shot NLI pipeline using typeform/distilbert-base-uncased-mnli. This record list has one column, prediction, where each record stores the predicted output for a sentence pair; each record is linked to the corresponding source item in glue_mrpc_validation. In this workflow, mrpc_zero_shot_prediction_value is the main prediction dataset used to convert top-ranked labels into binary predictions, evaluate accuracy and F1, inspect model behavior, and trace errors in the downstream mistakes and summary outputs."
embedding = get_embeddings(description)
vault.create_description("mrpc_zero_shot_prediction_value", description, embedding)

properties = {"dataset_type": "model predictions", "task": "paraphrase detection", "input_type": "sentence pair", "prediction_method": "zero-shot NLI classification", "model": "typeform/distilbert-base-uncased-mnli", "source": "glue/mrpc", "base_dataset": "glue_mrpc_validation", "benchmark": "GLUE", "split": "validation", "size": "408", "domain": "news", "labels": "paraphrase, not paraphrase", "output_granularity": "per-example prediction"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_zero_shot_prediction_value", cat, embedding, prop)



---[ TableVault Record ]---
---[ TableVault Record ]---



In [9]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))



---[ TableVault Record ]---
{'accuracy': 0.6299019607843137, 'f1': 0.7470686767169179}
                precision    recall  f1-score   support

not_paraphrase       0.38      0.26      0.31       129
    paraphrase       0.70      0.80      0.75       279

      accuracy                           0.63       408
     macro avg       0.54      0.53      0.53       408
  weighted avg       0.60      0.63      0.61       408

---[ TableVault Record ]---



In [10]:
for i in range(5):
    out = all_outputs[i]
    score_map = {label: float(score) for label, score in zip(out["labels"], out["scores"])}
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "top_label:", out["labels"][0])
    print("scores:", score_map)



---[ TableVault Record ]---
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 top_label: paraphrase
scores: {'paraphrase': 0.533607542514801, 'not paraphrase': 0.46639248728752136}
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 1 top_label: paraphrase
scores: {'paraphrase': 0.5474331974983215, 'not paraphrase': 0.45256683230400085}
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 

In [11]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

vault.create_record_list("zero_shot_nli_pipeline_mrpc_mistakes", column_names=["idx", "sentence1", "sentence2", "true", "pred"])

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

    vault.append_record("zero_shot_nli_pipeline_mrpc_mistakes", {
            "idx": int(i),
            "sentence1": sent1[i],
            "sentence2": sent2[i],
            "true": int(y_true[i]),
            "pred": int(y_pred[i]),
        },
        input_items = {
            "glue_mrpc_validation": [int(i), int(i)+ 1],
            "mrpc_zero_shot_prediction_value": [int(i), int(i)+ 1]
        }
    )

description = "This dataset stores a small error analysis subset for the zero-shot NLI paraphrase experiment on the GLUE MRPC validation split. It contains up to the first 10 validation examples where the model prediction disagreed with the ground-truth label.\n\nEach record corresponds to one misclassified sentence pair and has the following fields: idx (original example index in glue_mrpc_validation), sentence1, sentence2, true (ground-truth MRPC label: 0 = not paraphrase, 1 = paraphrase), and pred (predicted label from the zero-shot classifier).\n\nIts role in the workflow is to support qualitative inspection of model failures after generating predictions in mrpc_zero_shot_prediction_value and computing aggregate metrics. Records are linked back to the original validation examples and their corresponding prediction entries for traceability."
embedding = get_embeddings(description)
vault.create_description("zero_shot_nli_pipeline_mrpc_mistakes", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "error_analysis_subset", "content": "misclassified sentence pairs with true and predicted labels", "source": "glue/mrpc", "split": "validation", "selection": "first 10 mistakes", "model": "typeform/distilbert-base-uncased-mnli", "inference_method": "zero-shot nli pipeline", "labels": "paraphrase, not paraphrase", "size": "10"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("zero_shot_nli_pipeline_mrpc_mistakes", cat, embedding, prop)


---[ TableVault Record ]---
num_errors: 151
idx: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 1
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 1
idx: 3
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse a candidate before the primaries .
true: 1 pred: 0
idx: 9
sentence1: The results appear in the January issue of Cancer , an American Cancer Society journal , being published online toda

In [12]:
vault.create_record_list("zero_shot_nli_pipeline_mrpc_output", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("zero_shot_nli_pipeline_mrpc_output", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "mrpc_zero_shot_prediction_value": [0, len(ds)]
                    })

summary

description = "zero_shot_nli_pipeline_mrpc_output is the final summary dataset for this notebook\u2019s zero-shot paraphrase detection run on the GLUE MRPC validation set using the typeform/distilbert-base-uncased-mnli model. It contains aggregate evaluation results rather than per-example predictions, typically as a single record with three fields: accuracy (float), f1 (float), and classification_report (string containing the full sklearn classification report for the not_paraphrase and paraphrase classes). In this workflow, it serves as the compact experiment-level output that summarizes model performance over all validation examples and links back to the source MRPC validation data and the generated zero-shot prediction records."
embedding = get_embeddings(description)
vault.create_description("zero_shot_nli_pipeline_mrpc_output", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "evaluation summary", "method": "zero-shot nli classification", "model": "typeform/distilbert-base-uncased-mnli", "source": "glue/mrpc", "split": "validation", "size": "408", "input_type": "sentence pair", "labels": "paraphrase, not paraphrase", "metrics": "accuracy, f1, classification_report", "upstream_dataset": "glue_mrpc_validation", "prediction_dataset": "mrpc_zero_shot_prediction_value"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("zero_shot_nli_pipeline_mrpc_output", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [13]:
description = "This notebook runs a zero-shot paraphrase detection experiment on the GLUE MRPC validation set using the Hugging Face model typeform/distilbert-base-uncased-mnli. It retrieves sentence pairs and labels from TableVault, formats each pair as a two-sentence input, and uses a zero-shot NLI classification pipeline with candidate labels paraphrase and not paraphrase to predict whether the two sentences are semantically equivalent. The workflow stores per-example predictions in TableVault with lineage back to the source dataset, computes evaluation metrics including accuracy, F1, and a full classification report, inspects example outputs and misclassified cases, and saves mistake records and summary results as separate TableVault record lists. It also generates OpenAI text embeddings for descriptions and metadata so the dataset, outputs, errors, and overall notebook process can be documented and semantically indexed in TableVault." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("zero_shot_nli_pipeline_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "approach": "zero-shot classification via NLI", "model": "typeform/distilbert-base-uncased-mnli", "dataset": "glue/mrpc validation", "input_type": "sentence pair", "labels": "paraphrase, not paraphrase", "framework": "transformers pipeline", "library_stack": "PyTorch, Hugging Face Datasets, scikit-learn", "evaluation": "accuracy, f1-score, classification report", "error_analysis": "mistake extraction and inspection", "experiment_tracking": "TableVault", "embedding_model": "text-embedding-3-large", "device": "Apple MPS or CPU", "output_artifacts": "predictions, mistakes, summary metrics"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("zero_shot_nli_pipeline_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

